In [21]:
import sys
import pandas as pd
import geopandas as gpd
import numpy as np

from pathlib import Path

In [22]:
sys.path.append(str(Path.cwd().parent))

In [23]:
# Importa metodos
import src.data_utils as dt

In [25]:
import importlib

importlib.reload(dt)

<module 'src.data_utils' from 'c:\\Users\\simon\\Documents\\GitHub\\EtPilot\\src\\data_utils.py'>

In [2]:
# ============================================================
# 1. CONFIGURAÇÃO DOS CAMINHOS
# ============================================================

# Raiz do projeto

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Pasta principal dos dados do Censo 2022

CENSO_DIR = (
    PROJECT_ROOT
    / "data"
    / "censo_2022"
)


# ------------------------------------------------------------
# DADOS BÁSICOS DOS SETORES
# ------------------------------------------------------------

BASICOS_DIR = (
    CENSO_DIR
    / "agregados_basicos"
)

BASICOS_EXTRACTED_DIR = (
    BASICOS_DIR
    / "extracted"
)


# ------------------------------------------------------------
# RENDA POR SETORES
# ------------------------------------------------------------

RENDA_SETORES_DIR = (
    CENSO_DIR
    / "renda_setores"
)

RENDA_SETORES_EXTRACTED_DIR = (
    RENDA_SETORES_DIR
    / "extracted"
)


# ------------------------------------------------------------
# RENDA POR BAIRROS
# ------------------------------------------------------------

RENDA_BAIRROS_DIR = (
    CENSO_DIR
    / "renda_bairros"
)

RENDA_BAIRROS_EXTRACTED_DIR = (
    RENDA_BAIRROS_DIR
    / "extracted"
)


# ------------------------------------------------------------
# GEOMETRIA DOS SETORES
# ------------------------------------------------------------

SETORES_ZIP = (
    CENSO_DIR
    / "setores"
    / "setores_rs.zip"
)


# ------------------------------------------------------------
# GEOMETRIA DOS BAIRROS
# ------------------------------------------------------------

BAIRROS_ZIP = (
    CENSO_DIR
    / "bairros"
    / "bairros_rs.zip"
)


print("Raiz do projeto:")
print(PROJECT_ROOT)

print("\nCenso 2022:")
print(CENSO_DIR)

Raiz do projeto:
c:\Users\simon\Documents\GitHub\EtPilot

Censo 2022:
c:\Users\simon\Documents\GitHub\EtPilot\data\censo_2022


In [3]:
# ============================================================
# 2. IDENTIFICAÇÃO DE PORTO ALEGRE
# ============================================================

# IBGE de Porto Alegre.

COD_MUNICIPIO = "4314902"

In [4]:
# ============================================================
# 2. LOCALIZAÇÃO DOS ARQUIVOS
# ============================================================

# Eu procuro automaticamente o arquivo CSV de renda
# extraído dentro da pasta renda_setores/extracted.

arquivos_renda = list(
    RENDA_SETORES_EXTRACTED_DIR.glob("*.csv")
)


if len(arquivos_renda) == 0:
    raise FileNotFoundError(
        "Eu não encontrei nenhum CSV em "
        f"{RENDA_SETORES_EXTRACTED_DIR}"
    )


RENDA_SETOR_FILE = arquivos_renda[0]


print(
    "Arquivo de renda por setor:"
)

print(
    RENDA_SETOR_FILE
)

Arquivo de renda por setor:
c:\Users\simon\Documents\GitHub\EtPilot\data\censo_2022\renda_setores\extracted\Agregados_por_setores_renda_responsavel_BR.csv


In [5]:
# Eu procuro automaticamente o CSV com os
# agregados básicos do Censo 2022.

arquivos_basicos = list(
    BASICOS_EXTRACTED_DIR.glob("*.csv")
)


if len(arquivos_basicos) == 0:
    raise FileNotFoundError(
        "Eu não encontrei nenhum CSV em "
        f"{BASICOS_EXTRACTED_DIR}"
    )


SETOR_BASICO_FILE = arquivos_basicos[0]


print(
    "Arquivo de agregados básicos:"
)

print(
    SETOR_BASICO_FILE
)

Arquivo de agregados básicos:
c:\Users\simon\Documents\GitHub\EtPilot\data\censo_2022\agregados_basicos\extracted\Agregados_por_setores_basico_BR.csv


In [6]:
# ============================================================
# 3. VALIDAÇÃO DOS ARQUIVOS
# ============================================================

arquivos = {
    "Agregados básicos": SETOR_BASICO_FILE,
    "Renda por setores": RENDA_SETOR_FILE,
    "Geometria dos setores": SETORES_ZIP
}


for nome, arquivo in arquivos.items():

    print(
        f"{nome}:",
        "OK"
        if arquivo.exists()
        else "NÃO ENCONTRADO"
    )

    print(
        f"    {arquivo}"
    )

Agregados básicos: OK
    c:\Users\simon\Documents\GitHub\EtPilot\data\censo_2022\agregados_basicos\extracted\Agregados_por_setores_basico_BR.csv
Renda por setores: OK
    c:\Users\simon\Documents\GitHub\EtPilot\data\censo_2022\renda_setores\extracted\Agregados_por_setores_renda_responsavel_BR.csv
Geometria dos setores: OK
    c:\Users\simon\Documents\GitHub\EtPilot\data\censo_2022\setores\setores_rs.zip


In [7]:
# Eu carrego a base de rendimento dos responsáveis
# pelos domicílios.

renda = pd.read_csv(
    RENDA_SETOR_FILE,
    sep=";",
    dtype={
        "CD_SETOR": "string"
    },
    low_memory=False
)


# Eu seleciono somente os setores de Porto Alegre.

renda_poa = (
    renda[
        renda["CD_SETOR"]
        .str.startswith(
            COD_MUNICIPIO
        )
    ]
    .copy()
)


print(
    f"Setores de Porto Alegre com dados de renda: "
    f"{len(renda_poa):,}"
)

Setores de Porto Alegre com dados de renda: 2,713


In [9]:
renda_poa.head()
renda_poa.columns.tolist()

['CD_SETOR', 'V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']

In [11]:
# DADOS BÁSICOS DOS SETORES CENSITÁRIOS

basicos = pd.read_csv(
    SETOR_BASICO_FILE,
    sep=";",
    encoding="latin1",
    dtype={
        "CD_SETOR": "string"
    },
    low_memory=False
)

In [26]:
basicos_poa = dt.read_ibge_city(
    SETOR_BASICO_FILE,
    "4314902"
)

In [29]:
renda_poa = dt.read_ibge_city(
    RENDA_SETOR_FILE,
    "4314902"
)

renda_poa = renda_poa.rename(
    columns={
        "V06001": "responsaveis",
        "V06002": "residentes",
        "V06003": "variancia_moradores",
        "V06004": "renda_media_responsavel",
        "V06005": "variancia_renda"
    }
)

In [28]:
print(
    "Agregados básicos:"
)

print(
    basicos_poa.columns.tolist()
)


print(
    "\nRenda:"
)

print(
    renda_poa.columns.tolist()
)

Agregados básicos:
['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'v0001', 'v0002', 'v0003', 'v0004', 'v0005', 'v0006', 'v0007', 'v0008', 'v0009']

Renda:
['CD_SETOR', 'V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']
